In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1", device="cpu")

sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium."
]
embeddings = model.encode(sentences)

similarities = model.similarity(embeddings, embeddings)
print(similarities.shape)

In [ ]:
similarities


In [ ]:
embeddings = model.encode(sentences)
len(embeddings[0])

## Load Libraries

In [1]:
import json
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from tqdm.auto import tqdm

/home/jugal/miniconda3/envs/ds/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load the Text Chunks

In [2]:
# 1. Load chunk data
with open('../precessed_data/chunks/text_chunks_v1.json', 'r') as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks.")

Loaded 6974 chunks.


## Create Connection and Collection

In [3]:
# 2. Initialize Qdrant Client
client = QdrantClient(host="localhost", port=6333)
collection_name = "research_papers_v1"

# 3. Create collection if it doesn't exist
# We set size=1024 because mixedbread-ai/mxbai-embed-large-v1 outputs 1024-dimensional vectors
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
    )
    print(f"Created collection '{collection_name}'")

Created collection 'research_papers_v1'


## Define Embedding model

# Form embeddings and add in qdrant

In [8]:
# 4. Embed and upload in batches
batch_size = 64
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1", device="cpu")
print("Model Loaded.")
for i in tqdm(range(0, len(chunks), batch_size), desc="Uploading to Qdrant"):
    batch = chunks[i:i+batch_size]
    # print(batch[0])
    # break
    texts = [c['page_content'] for c in batch]
    
    # Generate embeddings using the model initialized in the first cell
    embeddings = model.encode(texts)
    
    batch_points = []
    for j, chunk in enumerate(batch):
        meta = chunk.get('metadata', {}).copy()
        
        # Remove large image base64 data from payload to save space
        if 'images' in meta:
            del meta['images']
            
        batch_points.append(
            PointStruct(
                id=i+j,
                vector=embeddings[j].tolist(),
                payload={
                    "text": chunk['page_content'],
                    **meta,
                    "chunk_id": chunk.get('chunk_id', ''),
                    "chunk_index": chunk.get('chunk_index', 0)
                }
            )
        )
    
    client.upsert(
        collection_name=collection_name,
        points=batch_points
    )

print("Finished uploading to Qdrant!")


Model Loaded.


Uploading to Qdrant: 100%|██████████| 109/109 [3:30:07<00:00, 115.66s/it] 

Finished uploading to Qdrant!


Latency = 210m on cpu

# Image Embeddings 

In [1]:
import json
import numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct
from tqdm.auto import tqdm

/home/jugal/miniconda3/envs/ds/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1782138003.880119   52350 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782138003.925033   52350 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782138005.155987   52350 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to

In [2]:
# 1. Load image chunks data
with open('../precessed_data/chunks/image_chunks_v1.json', 'r') as f:
    image_chunks = json.load(f)

print(f"Loaded {len(image_chunks)} image chunks.")

Loaded 756 image chunks.


In [ ]:
# 2. Initialize CLIP model 
# We use clip-ViT-B-32 which outputs 512-dimensional vectors
clip_model = SentenceTransformer('clip-ViT-B-32', device='cpu')

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [ ]:
# 3. Initialize Qdrant Client
client = QdrantClient(host="localhost", port=6333)
collection_name = "research_papers_v1"

In [ ]:
# 4. Embed and upload in batches
batch_size = 32

for i in tqdm(range(0, len(image_chunks), batch_size), desc="Uploading Images to Qdrant"):
    batch = image_chunks[i:i+batch_size]
    
    # Load images using PIL
    images = []
    valid_batch = []
    for chunk in batch:
        try:
            img = Image.open(chunk['image_path'])
            images.append(img)
            valid_batch.append(chunk)
        except Exception as e:
            print(f"Error loading {chunk['image_path']}: {e}")
            continue
            
    if not images:
        continue
        
    # Generate embeddings
    embeddings = clip_model.encode(images)
    
    batch_points = []
    for j, chunk in enumerate(valid_batch):
        # Zero-pad the 512-dim CLIP vector to 1024-dim to fit the existing collection
        padded_vector = np.pad(embeddings[j], (0, 1024 - len(embeddings[j])), 'constant').tolist()
        
        # Create a unique ID to avoid overwriting text chunks. 
        # We start from a high offset (1,000,000) for safety.
        unique_id = 1000000 + i + j
        
        batch_points.append(
            PointStruct(
                id=unique_id,
                vector=padded_vector,
                payload={
                    "image_path": chunk['image_path'],
                    **chunk['metadata'],
                    "chunk_id": chunk.get('chunk_id', ''),
                    "type": "image_chunk" # Add a type to filter easily later
                }
            )
        )
    
    client.upsert(
        collection_name=collection_name,
        points=batch_points
    )

print("Finished uploading image embeddings to Qdrant!")


Uploading Images to Qdrant: 100%|██████████| 24/24 [00:38<00:00,  1.59s/it]

Finished uploading image embeddings to Qdrant!


# Getting the Exbeddings out of the vector store

In [5]:
from qdrant_client import QdrantClient
from pathlib import Path
import json
import numpy as np


HOST = "localhost"
PORT = 6333
COLLECTION_NAME = "research_papers_v1"

# Set this to your dense vector name if the collection uses named vectors.
# If the collection has only one unnamed dense vector, keep it as None.
DENSE_VECTOR_NAME = None   # example: "dense"

BATCH_SIZE = 256
OUT_FILE = Path("qdrant_dense_embeddings.jsonl")


def extract_dense_vector(point, dense_vector_name=None):
    """
    Returns the dense vector from a Qdrant point.

    Works for:
    1) unnamed single-vector collections -> point.vector is a list
    2) named-vector collections -> point.vector is a dict
    """
    vec = point.vector

    # Named vectors: point.vector is usually a dict like {"dense": [...], "sparse": {...}}
    if isinstance(vec, dict):
        if dense_vector_name is None:
            # If you did not specify a name, try to take the first vector.
            # Use this only if you know the collection stores just one dense vector.
            dense_vector = next(iter(vec.values()))
        else:
            if dense_vector_name not in vec:
                raise KeyError(
                    f"Vector name '{dense_vector_name}' not found. "
                    f"Available vectors: {list(vec.keys())}"
                )
            dense_vector = vec[dense_vector_name]
    else:
        # Unnamed single vector
        dense_vector = vec

    return np.asarray(dense_vector, dtype=np.float32)


def export_dense_embeddings():
    client = QdrantClient(host=HOST, port=PORT)

    offset = None
    total = 0

    with OUT_FILE.open("w", encoding="utf-8") as f:
        while True:
            points, next_offset = client.scroll(
                collection_name=COLLECTION_NAME,
                offset=offset,
                limit=BATCH_SIZE,
                with_payload=True,
                with_vectors=True,
            )

            if not points:
                break

            for point in points:
                dense_vec = extract_dense_vector(point, DENSE_VECTOR_NAME)

                payload = point.payload or {}

                record = {
                    "point_id": point.id,
                    "payload": payload,
                    "vector": dense_vec.tolist(),
                }

                f.write(json.dumps(record) + "\n")
                total += 1

            offset = next_offset
            if offset is None:
                break

    print(f"Exported {total} vectors to {OUT_FILE}")


if __name__ == "__main__":
    export_dense_embeddings()

Exported 7730 vectors to qdrant_dense_embeddings.jsonl


# Building Hybrid VectorStore

In [3]:
from qdrant_client import QdrantClient
from qdrant_client import models
from fastembed import SparseTextEmbedding
from tqdm.auto import tqdm

# 1. Initialize the BM25 model for sparse embeddings
# Note: This might download the BM25 model weights on the first run
bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")

# 2. Setup the Qdrant Client (Using local storage in a folder)
client = QdrantClient(host="localhost", port=6333)

collection_name = "research_papers_v2_hybrid"

# 3. Create a collection to hold BOTH Dense and Sparse vectors
dense_vector_size = len(dense_embed[0]["vector"])

if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config={
            "dense": models.VectorParams(
                size=dense_vector_size, 
                distance=models.Distance.COSINE
            )
        },
        sparse_vectors_config={
            "sparse": models.SparseVectorParams()
        }
    )

# 4. Prepare and generate vectors
points = []

# Extract all text payloads to generate BM25 embeddings in batch
print("Generating BM25 Sparse Embeddings...")
texts = [item["payload"].get("text", "") for item in dense_embed]
sparse_embeddings = list(bm25_model.embed(texts))

print("Preparing Qdrant Points...")
for i, item in enumerate(tqdm(dense_embed)):
    point_id = item["point_id"]
    payload = item["payload"]
    dense_vector = item["vector"]
    
    # Get the generated sparse embedding
    sparse_emb = sparse_embeddings[i]
    
    # Convert FastEmbed's format to Qdrant's SparseVector format
    sparse_vector = models.SparseVector(
        indices=sparse_emb.indices.tolist(),
        values=sparse_emb.values.tolist()
    )
    
    # Create the point struct with named vectors ('dense' and 'sparse')
    point = models.PointStruct(
        id=point_id,
        payload=payload,
        vector={
            "dense": dense_vector,
            "sparse": sparse_vector
        }
    )
    points.append(point)

# 5. Upload the points to the collection in batches
print(f"Uploading {len(points)} points to Qdrant...")
client.upload_points(
    collection_name=collection_name,
    points=points,
    batch_size=100,
    wait=True
)
print("Vector store successfully built with dense and sparse BM25 vectors!")


/home/jugal/miniconda3/envs/ds/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Generating BM25 Sparse Embeddings...
Preparing Qdrant Points...


100%|██████████| 7730/7730 [00:00<00:00, 13730.19it/s]


Uploading 7730 points to Qdrant...
Vector store successfully built with dense and sparse BM25 vectors!


1 minute for sparse vectors

# Colbert 

In [1]:
from fastembed import LateInteractionTextEmbedding

model = LateInteractionTextEmbedding(
    model_name="colbert-ir/colbertv2.0"
)

/home/jugal/miniconda3/envs/ds/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [01:54<00:00, 22.83s/it]


In [3]:

text = """Elon Reeve Musk (/ˈiːlɒn/ ⓘ EE-lon; born June 28, 1971) is a businessman and former public official who is the CEO and largest shareholder of Tesla and SpaceX. Musk has been the wealthiest person in the world since 2025, and became the first and only trillionaire in terms of US dollars in mid June 2026; however as of June 21, 2026, Forbes estimates his net worth to be US$951 billion.

Born into the wealthy Musk family in Pretoria, South Africa, Musk emigrated in 1989 to Canada; he has Canadian citizenship since his mother was born there. He received bachelor's degrees in 1997 from the University of Pennsylvania before moving to California to pursue business ventures. In 1995, Musk co-founded Zip2, a web software company. Following its sale in 1999, he co-founded X.com, an e-commerce payment system that merged with Confinity in March 2000 to form PayPal, which was acquired by eBay in 2002. Musk also became an American citizen in 2002.

In 2002, Musk founded and became CEO and chief engineer of SpaceX, a space technology company; the company has since led innovations in reusable rockets and commercial spaceflight. Musk joined Tesla as an early investor in 2004 and became its CEO and product architect in 2008; it has since become a leader in electric vehicles. In 2015, Musk co-founded OpenAI to advance artificial intelligence (AI) research, but later left; his growing discontent with the organization's direction and leadership in the AI boom in the 2020s led him to establish xAI, which became a subsidiary of SpaceX in 2026. In 2022, he acquired Twitter, a social networking service; he implemented significant changes and rebranded it as X in 2023. His other businesses include Neuralink, a neurotechnology company that he co-founded in 2016, and the Boring Company, a tunneling company that he founded in 2017. In November 2025, Tesla approved a pay package worth $1 trillion for Musk, which he is to receive over 10 years if certain milestones are met, such as achieving a market capitalization of $8.5 trillion.

Musk is a supporter of global far-right politics, figures, and political parties. He was the largest donor in the 2024 U.S. presidential election, where he supported Donald Trump. After Trump was inaugurated in January 2025, Musk served as Senior Advisor to the President and as the de facto head of the Department of Government Efficiency (DOGE). Musk left the Trump administration in May 2025 and returned to managing his companies; shortly thereafter he had a public feud with Trump.

Musk's political activities, statements and views have made him a polarizing figure. He has been criticized for making unscientific and misleading statements, including spreading COVID-19 misinformation, promoting conspiracy theories, and affirming antisemitic, white nationalist, racist, and transphobic comments. His acquisition of Twitter was controversial because, following his pledge to decrease censorship, there was an increase in hate speech and misinformation on the service. His role in the second Trump administration attracted public backlash, particularly in response to DOGE and its cuts to the US Agency for International Development (USAID).
"""

# query = "When does elon musk created SpaseX"
text = text[:1500]

documents = [text]


In [6]:
import time

start = time.time()
doc_embeddings = list(model.passage_embed(documents))
print(time.time() - start)

0.2318723201751709


In [8]:
0.34 * 7500 / 60

42.5

In [5]:
doc_embeddings

[array([[ 0.15329435, -0.04323614, -0.17336755, ...,  0.17695992,
         -0.08424366, -0.02156661],
        [ 0.1138189 , -0.05338327, -0.16601162, ...,  0.13968766,
         -0.08977207,  0.00775665],
        [-0.0758727 ,  0.06728075,  0.0310065 , ...,  0.06023695,
          0.04987186,  0.07672783],
        ...,
        [-0.02110421, -0.17998847, -0.07041971, ..., -0.00368085,
          0.00937243,  0.0580478 ],
        [-0.03504158, -0.19714859, -0.0115671 , ...,  0.02637375,
          0.0179191 ,  0.0796407 ],
        [ 0.09557072, -0.09378482, -0.20182176, ...,  0.16883801,
         -0.04490271,  0.03191827]], shape=(297, 128), dtype=float32)]

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client import models
from fastembed import LateInteractionTextEmbedding, SparseTextEmbedding, ImageEmbedding
import uuid

def initialize_multimodal_vectorstore(text_chunks, image_paths=None):
    """
    Creates a Qdrant collection configured for ColBERT, BM25, and CLIP, 
    and inserts the generated embeddings.
    """
    # 1. Connect to Qdrant Client
    client = QdrantClient(host="localhost", port=6333)
    collection_name = "research_papers_v3_Colbert"
    
    # 2. Recreate Collection with multi-vector configurations
    client.recreate_collection(
        collection_name=collection_name,
        vectors_config={
            # Late Interaction Vector for Text via ColBERTv2 (128-dim)
            "colbert": models.VectorParams(
                size=128,
                distance=models.Distance.COSINE,
                multivector_config=models.MultiVectorConfig(
                    comparator=models.MultiVectorComparator.MAX_SIM
                ),
            ),
            # Dense Vector for Images via CLIP (512-dim for ViT-B-32)
            "clip": models.VectorParams(
                size=512,  
                distance=models.Distance.COSINE,
            ),
        },
        sparse_vectors_config={
            # Sparse Vector for BM25
            "bm25": models.SparseVectorParams(
                modifier=models.Modifier.IDF
            ),
        }
    )
    print(f"Collection '{collection_name}' successfully created.")

    # 3. Initialize fastembed models
    print("Initializing embedding models...")
    colbert_model = LateInteractionTextEmbedding(model_name="colbert-ir/colbertv2.0")
    bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")
    # Using CLIP vision model for images
    clip_model = ImageEmbedding(model_name="Qdrant/clip-ViT-B-32-vision")

    # 4. Generate Embeddings
    print("Generating embeddings (this may take a moment)...")
    
    # Generate ColBERT and BM25 embeddings from the text chunks
    colbert_embeddings = list(colbert_model.embed(text_chunks))
    bm25_embeddings = list(bm25_model.embed(text_chunks))
    
    # Generate CLIP embeddings if image paths are provided (falling back to empty lists if not)
    if image_paths and len(image_paths) == len(text_chunks):
        clip_embeddings = list(clip_model.embed(image_paths))
    else:
        # Dummy placeholder to avoid breaking the loop if you don't have images for every chunk yet
        clip_embeddings = [[] for _ in text_chunks]

    # 5. Prepare and Upload Points
    points = []
    for idx in range(len(text_chunks)):
        vector_payload = {
            "colbert": colbert_embeddings[idx].tolist() if hasattr(colbert_embeddings[idx], 'tolist') else colbert_embeddings[idx],
            "bm25": models.SparseVector(
                indices=bm25_embeddings[idx].indices.tolist() if hasattr(bm25_embeddings[idx].indices, 'tolist') else bm25_embeddings[idx].indices,
                values=bm25_embeddings[idx].values.tolist() if hasattr(bm25_embeddings[idx].values, 'tolist') else bm25_embeddings[idx].values,
            )
        }
        
        # Add CLIP dense vectors only if we successfully created them
        if clip_embeddings[idx] is not None and len(clip_embeddings[idx]) > 0:
            vector_payload["clip"] = clip_embeddings[idx].tolist() if hasattr(clip_embeddings[idx], 'tolist') else clip_embeddings[idx]

        points.append(
            models.PointStruct(
                id=str(uuid.uuid4()),
                payload={
                    "text": text_chunks[idx],
                    "image_path": image_paths[idx] if image_paths else None
                },
                vector=vector_payload
            )
        )

    # 6. Insert points into Qdrant
    client.upload_points(
        collection_name=collection_name,
        points=points
    )
    
    print(f"Successfully inserted {len(points)} points into '{collection_name}'.")
    return client

# --- Example Usage ---
# sample_texts = ["Research paper intro abstract here...", "Methodology detailed explanation..."]
# sample_images = ["./images/fig1.png", "./images/fig2.png"]
# qdrant_client = initialize_multimodal_vectorstore(sample_texts, sample_images)


# Loading Data

In [ ]:
import json

# 1. Load image chunks data
with open('../precessed_data/chunks/image_chunks_v1.json', 'r') as f:
    image_chunks = json.load(f)

print(f"Loaded {len(image_chunks)} image chunks.")

# 1. Load chunk data
with open('../precessed_data/chunks/text_chunks_v1.json', 'r') as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} text chunks.")

import json

with open("/home/jugal/Documents/DataScience/Advanced_RAG/Research_Paper_rag/Experiment_pipeline/qdrant_dense_embeddings.jsonl", "r") as f:
    # Read line by line and parse each line individually
    dense_embed = [json.loads(line) for line in f]

# Check how many embeddings were loaded
print(f"Loaded {len(dense_embed)} embeddings.")

image_embeddings = []

for i in dense_embed:
    if "image_path" in i['payload']:
        image_embeddings.append(i)

print(len(image_embeddings))


Loaded 756 image chunks.


In [7]:
from qdrant_client import QdrantClient
from qdrant_client import models
from fastembed import LateInteractionTextEmbedding, SparseTextEmbedding
import uuid
import time
from tqdm.auto import tqdm

def build_vectorstore(chunks, image_embeddings, batch_size=16): # Reduced default batch_size
    """
    Creates the Qdrant VectorStore. Computes ColBERT and BM25 embeddings 
    for text chunks in batches, and inserts them alongside pre-computed image_embeddings.
    """
    client = QdrantClient(host="localhost", port=6333)
    collection_name = "research_papers_v3_Colbert"
    
    # 1. Recreate Collection with multi-vector configurations
    if client.collection_exists(collection_name):
        client.delete_collection(collection_name)
        
    client.create_collection(
        collection_name=collection_name,
        vectors_config={
            "colbert": models.VectorParams(
                size=128,
                distance=models.Distance.COSINE,
                multivector_config=models.MultiVectorConfig(
                    comparator=models.MultiVectorComparator.MAX_SIM
                ),
            ),
            "clip": models.VectorParams(
                size=512,  
                distance=models.Distance.COSINE,
            ),
        },
        sparse_vectors_config={
            "bm25": models.SparseVectorParams(
                modifier=models.Modifier.IDF
            ),
        }
    )
    print(f"Collection '{collection_name}' successfully created.")

    # 2. Initialize fastembed text models
    print("Initializing text embedding models...")
    colbert_model = LateInteractionTextEmbedding(model_name="colbert-ir/colbertv2.0")
    bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")

    # 3 & 4. Generate and Upload Embeddings for Text Chunks in Batches
    print("Generating embeddings and uploading text chunks in batches...")
    for i in tqdm(range(0, len(chunks), batch_size), desc="Text Chunks"):
        batch_chunks = chunks[i:i+batch_size]
        texts = [chunk['page_content'] for chunk in batch_chunks]
        
        colbert_batch = list(colbert_model.embed(texts))
        bm25_batch = list(bm25_model.embed(texts))

        points = []
        for j, chunk in enumerate(batch_chunks):
            payload = chunk.get('metadata', {}).copy()
            
            # --- CRITICAL: Remove large images from metadata to save space ---
            if 'images' in payload:
                del payload['images']
                
            payload['page_content'] = chunk['page_content']
            payload['type'] = 'text_chunk'
            
            points.append(
                models.PointStruct(
                    id=str(uuid.uuid4()), 
                    payload=payload,
                    vector={
                        "colbert": colbert_batch[j].tolist() if hasattr(colbert_batch[j], 'tolist') else colbert_batch[j],
                        "bm25": models.SparseVector(
                            indices=bm25_batch[j].indices.tolist() if hasattr(bm25_batch[j].indices, 'tolist') else bm25_batch[j].indices,
                            values=bm25_batch[j].values.tolist() if hasattr(bm25_batch[j].values, 'tolist') else bm25_batch[j].values,
                        )
                    }
                )
            )
            
        client.upload_points(
            collection_name=collection_name,
            points=points,
            wait=True
        )

    # 5 & 6. Upload Image Embeddings in Batches
    print("Uploading image embeddings in batches...")
    for i in tqdm(range(0, len(image_embeddings), batch_size), desc="Image Embeddings"):
        batch_images = image_embeddings[i:i+batch_size]
        points = []
        
        for img_embed in batch_images:
            # Also clean up image payloads just in case
            payload = img_embed['payload'].copy()
            if 'images' in payload:
                del payload['images']
                
            points.append(
                models.PointStruct(
                    id=img_embed['point_id'],
                    payload=payload,
                    vector={
                        "clip": img_embed['vector']
                    }
                )
            )
            
        client.upload_points(
            collection_name=collection_name,
            points=points,
            wait=True
        )

    print("Successfully built the VectorStore!")
    return client

# --- Example Usage ---
start = time.time()
# Note: Lower batch_size to 8 or 16 to keep payloads under Qdrant's 32MB limit
qdrant_client = build_vectorstore(chunks, image_embeddings, batch_size=8)
print(f"Time taken: {time.time() - start:.2f} seconds")


Collection 'research_papers_v3_Colbert' successfully created.
Initializing text embedding models...
Generating embeddings and uploading text chunks in batches...


Text Chunks: 100%|██████████| 872/872 [1:26:18<00:00,  5.94s/it]


Uploading image embeddings in batches...


Image Embeddings:   0%|          | 0/95 [00:00<?, ?it/s]/tmp/ipykernel_149709/2298701631.py:110: UserWarning: Batch upload failed 1 times. Retrying...
  client.upload_points(
/tmp/ipykernel_149709/2298701631.py:110: UserWarning: Batch upload failed 2 times. Retrying...
  client.upload_points(
/tmp/ipykernel_149709/2298701631.py:110: UserWarning: Batch upload failed 3 times. Retrying...
  client.upload_points(
Image Embeddings:   0%|          | 0/95 [00:00<?, ?it/s]


UnexpectedResponse: Unexpected Response: 400 (Bad Request)
Raw response content:
b'{"status":{"error":"Wrong input: Vector dimension error: expected dim: 512, got 1024 for vector \'clip\'"},"time":0.000032599}'

86 minutes for the approx 7000 emebddings

In [8]:
from qdrant_client import QdrantClient
from qdrant_client import models
from tqdm.auto import tqdm

# Connect to the existing Qdrant client
client = QdrantClient(host="localhost", port=6333)
collection_name = "research_papers_v3_Colbert"
batch_size = 16

print("Uploading image embeddings to existing collection...")

for i in tqdm(range(0, len(image_embeddings), batch_size), desc="Image Embeddings"):
    batch_images = image_embeddings[i:i+batch_size]
    points = []
    
    for img_embed in batch_images:
        # Clean up payload
        payload = img_embed['payload'].copy()
        if 'images' in payload:
            del payload['images']
            
        # CRITICAL FIX: The stored image vectors are 1024-dim (from your previous zero-padding).
        # We slice [:512] to get the original 512-dim CLIP vector back.
        original_clip_vector = img_embed['vector'][:512]
            
        points.append(
            models.PointStruct(
                id=img_embed['point_id'],
                payload=payload,
                vector={
                    "clip": original_clip_vector
                }
            )
        )
        
    client.upload_points(
        collection_name=collection_name,
        points=points,
        wait=True
    )

print("Finished uploading image embeddings!")


Uploading image embeddings to existing collection...


Image Embeddings: 100%|██████████| 48/48 [00:01<00:00, 24.66it/s]

Finished uploading image embeddings!


# Trying for the Diff Embedding Models

In [15]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("WhereIsAI/UAE-Large-V1", device = "cpu")

sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium."
]
embeddings = model.encode(sentences)

similarities = model.similarity(embeddings, embeddings)
print(similarities.shape)

torch.Size([3, 3])


In [18]:
embeddings[0].shape

(1024,)

In [13]:
from qdrant_client import QdrantClient
from qdrant_client import models
from fastembed import SparseTextEmbedding
from sentence_transformers import SentenceTransformer
import uuid
import time
from tqdm.auto import tqdm

def build_vectorstore(chunks, image_embeddings, batch_size=16):
    """
    Creates the Qdrant VectorStore. Computes UAE-Large-V1 and BM25 embeddings 
    for text chunks in batches, and inserts them alongside pre-computed image_embeddings.
    """
    client = QdrantClient(host="localhost", port=6333)
    collection_name = "research_papers_v4_UAE"
    
    # 1. Recreate Collection with multi-vector configurations
    if client.collection_exists(collection_name):
        client.delete_collection(collection_name)
        
    client.create_collection(
        collection_name=collection_name,
        vectors_config={
            "uae": models.VectorParams(
                size=1024,
                distance=models.Distance.COSINE,
            ),
            "clip": models.VectorParams(
                size=512,  
                distance=models.Distance.COSINE,
            ),
        },
        sparse_vectors_config={
            "bm25": models.SparseVectorParams(
                modifier=models.Modifier.IDF
            ),
        }
    )
    print(f"Collection '{collection_name}' successfully created.")

    # 2. Initialize text models
    print("Initializing text embedding models...")
    uae_model = SentenceTransformer("WhereIsAI/UAE-Large-V1", device="cpu")
    bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")

    # 3 & 4. Generate and Upload Embeddings for Text Chunks in Batches
    print("Generating embeddings and uploading text chunks in batches...")
    for i in tqdm(range(0, len(chunks), batch_size), desc="Text Chunks"):
        batch_chunks = chunks[i:i+batch_size]
        texts = [chunk['page_content'] for chunk in batch_chunks]
        
        # Generate dense embeddings using UAE-Large-V1
        uae_batch = uae_model.encode(texts)
        bm25_batch = list(bm25_model.embed(texts))

        points = []
        for j, chunk in enumerate(batch_chunks):
            payload = chunk.get('metadata', {}).copy()
            
            # --- CRITICAL: Remove large images from metadata to save space ---
            if 'images' in payload:
                del payload['images']
                
            payload['page_content'] = chunk['page_content']
            payload['type'] = 'text_chunk'
            
            points.append(
                models.PointStruct(
                    id=str(uuid.uuid4()), 
                    payload=payload,
                    vector={
                        "uae": uae_batch[j].tolist(),
                        "bm25": models.SparseVector(
                            indices=bm25_batch[j].indices.tolist() if hasattr(bm25_batch[j].indices, 'tolist') else bm25_batch[j].indices,
                            values=bm25_batch[j].values.tolist() if hasattr(bm25_batch[j].values, 'tolist') else bm25_batch[j].values,
                        )
                    }
                )
            )
            
        client.upload_points(
            collection_name=collection_name,
            points=points,
            wait=True
        )

    # 5 & 6. Upload Image Embeddings in Batches
    # We directly use the image_embeddings provided so we don't calculate them again.
    print("Uploading image embeddings in batches...")
    for i in tqdm(range(0, len(image_embeddings), batch_size), desc="Image Embeddings"):
        batch_images = image_embeddings[i:i+batch_size]
        points = []
        
        for img_embed in batch_images:
            # Also clean up image payloads just in case
            payload = img_embed['payload'].copy()
            if 'images' in payload:
                del payload['images']
                
            points.append(
                models.PointStruct(
                    id=img_embed['point_id'],
                    payload=payload,
                    vector={
                        "clip": img_embed['vector']
                    }
                )
            )
            
        client.upload_points(
            collection_name=collection_name,
            points=points,
            wait=True
        )

    print("Successfully built the VectorStore!")
    return client

# --- Example Usage ---
start = time.time()
# Note: Lower batch_size to 8 or 16 to keep payloads under Qdrant's 32MB limit
qdrant_client = build_vectorstore(chunks, image_embeddings, batch_size=8)
print(f"Time taken: {time.time() - start:.2f} seconds")

Collection 'research_papers_v4_UAE' successfully created.
Initializing text embedding models...
Generating embeddings and uploading text chunks in batches...


Text Chunks: 100%|██████████| 872/872 [3:10:03<00:00, 13.08s/it]  


Uploading image embeddings in batches...


Image Embeddings:   0%|          | 0/95 [00:00<?, ?it/s]/tmp/ipykernel_84241/1332567269.py:110: UserWarning: Batch upload failed 1 times. Retrying...
  client.upload_points(
/tmp/ipykernel_84241/1332567269.py:110: UserWarning: Batch upload failed 2 times. Retrying...
  client.upload_points(
/tmp/ipykernel_84241/1332567269.py:110: UserWarning: Batch upload failed 3 times. Retrying...
  client.upload_points(
Image Embeddings:   0%|          | 0/95 [00:00<?, ?it/s]


UnexpectedResponse: Unexpected Response: 400 (Bad Request)
Raw response content:
b'{"status":{"error":"Wrong input: Vector dimension error: expected dim: 512, got 1024 for vector \'clip\'"},"time":0.000021694}'

Latency : 190 min

In [14]:
from qdrant_client import QdrantClient
from qdrant_client import models
from tqdm.auto import tqdm

# Connect to the existing Qdrant client
client = QdrantClient(host="localhost", port=6333)
collection_name = "research_papers_v4_UAE"
batch_size = 16

print("Uploading image embeddings to existing collection...")

for i in tqdm(range(0, len(image_embeddings), batch_size), desc="Image Embeddings"):
    batch_images = image_embeddings[i:i+batch_size]
    points = []
    
    for img_embed in batch_images:
        # Clean up payload
        payload = img_embed['payload'].copy()
        if 'images' in payload:
            del payload['images']
            
        # CRITICAL FIX: The stored image vectors are 1024-dim (from your previous zero-padding).
        # We slice [:512] to get the original 512-dim CLIP vector back.
        original_clip_vector = img_embed['vector'][:512]
            
        points.append(
            models.PointStruct(
                id=img_embed['point_id'],
                payload=payload,
                vector={
                    "clip": original_clip_vector
                }
            )
        )
        
    client.upload_points(
        collection_name=collection_name,
        points=points,
        wait=True
    )

print("Finished uploading image embeddings!")


Uploading image embeddings to existing collection...


Image Embeddings: 100%|██████████| 48/48 [00:00<00:00, 58.71it/s]

Finished uploading image embeddings!


In [20]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-large-en-v1.5", device="cpu")

sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium."
]
embeddings = model.encode(sentences)
# print(len(enbeddings[0]))
similarities = model.similarity(embeddings, embeddings)
print(similarities.shape)

torch.Size([3, 3])


In [21]:
embeddings.shape

(3, 1024)

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client import models
from fastembed import SparseTextEmbedding
from sentence_transformers import SentenceTransformer
import uuid
import time
from tqdm.auto import tqdm

def build_vectorstore(chunks, image_embeddings, batch_size=16):
    """
    Creates the Qdrant VectorStore. Computes UAE-Large-V1 and BM25 embeddings 
    for text chunks in batches, and inserts them alongside pre-computed image_embeddings.
    """
    client = QdrantClient(host="localhost", port=6333)
    collection_name = "research_papers_v4_bge"
    
    # 1. Recreate Collection with multi-vector configurations
    if client.collection_exists(collection_name):
        client.delete_collection(collection_name)
        
    client.create_collection(
        collection_name=collection_name,
        vectors_config={
            "uae": models.VectorParams(
                size=1024,
                distance=models.Distance.COSINE,
            ),
            "clip": models.VectorParams(
                size=512,  
                distance=models.Distance.COSINE,
            ),
        },
        sparse_vectors_config={
            "bm25": models.SparseVectorParams(
                modifier=models.Modifier.IDF
            ),
        }
    )
    print(f"Collection '{collection_name}' successfully created.")

    # 2. Initialize text models
    print("Initializing text embedding models...")
    uae_model = SentenceTransformer("WhereIsAI/UAE-Large-V1", device="cpu")
    bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")

    # 3 & 4. Generate and Upload Embeddings for Text Chunks in Batches
    print("Generating embeddings and uploading text chunks in batches...")
    for i in tqdm(range(0, len(chunks), batch_size), desc="Text Chunks"):
        batch_chunks = chunks[i:i+batch_size]
        texts = [chunk['page_content'] for chunk in batch_chunks]
        
        # Generate dense embeddings using UAE-Large-V1
        uae_batch = uae_model.encode(texts)
        bm25_batch = list(bm25_model.embed(texts))

        points = []
        for j, chunk in enumerate(batch_chunks):
            payload = chunk.get('metadata', {}).copy()
            
            # --- CRITICAL: Remove large images from metadata to save space ---
            if 'images' in payload:
                del payload['images']
                
            payload['page_content'] = chunk['page_content']
            payload['type'] = 'text_chunk'
            
            points.append(
                models.PointStruct(
                    id=str(uuid.uuid4()), 
                    payload=payload,
                    vector={
                        "uae": uae_batch[j].tolist(),
                        "bm25": models.SparseVector(
                            indices=bm25_batch[j].indices.tolist() if hasattr(bm25_batch[j].indices, 'tolist') else bm25_batch[j].indices,
                            values=bm25_batch[j].values.tolist() if hasattr(bm25_batch[j].values, 'tolist') else bm25_batch[j].values,
                        )
                    }
                )
            )
            
        client.upload_points(
            collection_name=collection_name,
            points=points,
            wait=True
        )

    # 5 & 6. Upload Image Embeddings in Batches
    # We directly use the image_embeddings provided so we don't calculate them again.
    print("Uploading image embeddings in batches...")
    for i in tqdm(range(0, len(image_embeddings), batch_size), desc="Image Embeddings"):
        batch_images = image_embeddings[i:i+batch_size]
        points = []
        
        for img_embed in batch_images:
            # Also clean up image payloads just in case
            payload = img_embed['payload'].copy()
            if 'images' in payload:
                del payload['images']
                
            points.append(
                models.PointStruct(
                    id=img_embed['point_id'],
                    payload=payload,
                    vector={
                        "clip": img_embed['vector']
                    }
                )
            )
            
        client.upload_points(
            collection_name=collection_name,
            points=points,
            wait=True
        )

    print("Successfully built the VectorStore!")
    return client

# --- Example Usage ---
start = time.time()
# Note: Lower batch_size to 8 or 16 to keep payloads under Qdrant's 32MB limit
qdrant_client = build_vectorstore(chunks, image_embeddings, batch_size=8)
print(f"Time taken: {time.time() - start:.2f} seconds")

Collection 'research_papers_v4_bge' successfully created.
Initializing text embedding models...
Generating embeddings and uploading text chunks in batches...


Text Chunks:  60%|██████    | 527/872 [1:42:51<1:06:45, 11.61s/it]